# IPL Points Table Analysis & 2025 Prediction

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

In [2]:
# colors setup 

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor': '#161616',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#e0e0e0',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': '#ffffff',
    'axes.labelsize': 10,
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'text.color': '#e0e0e0',
    'grid.color': '#2a2a2a',
    'grid.linewidth': 0.6,
    'legend.facecolor': '#1a1a1a',
    'legend.edgecolor': '#333',
    'legend.labelcolor': '#e0e0e0',
    'legend.fontsize': 8,
    'savefig.facecolor': '#0d0d0d'
})
sns.set_theme(style='dark', rc={
    'axes.facecolor': '#161616',
    'figure.facecolor': '#0d0d0d',
    'grid.color': '#2a2a2a'
})

TEAM_COLORS = {
    'Mumbai Indians'                    : '#004BA0',
    'Chennai Super Kings'               : '#F9CD05',
    'Royal Challengers Bangalore'       : '#EC1C24',
    'Kolkata Knight Riders'             : '#3A225D',
    'Sunrisers Hyderabad'               : '#F7A721',
    'Delhi Capitals'                    : '#0078BC',
    'Delhi Daredevils'                  : '#0078BC',
    'Rajasthan Royals'                  : '#EA1A85',
    'Kings XI Punjab'                   : '#AADA18',
    'Punjab Kings'                      : '#AADA18',
    'Deccan Chargers'                   : '#FF6E00',
    'Kochi Tuskers Kerala'              : '#00A85A',
    'Pune Warriors'                     : '#1C4E9D',
    'Rising Pune Supergiant'            : '#6C2D91',
    'Rising Pune Supergiants'           : '#6C2D91',
    'Gujarat Lions'                     : '#E35020',
    'Lucknow Super Giants'              : '#A0E6FF',
    'Gujarat Titans'                    : '#1D3461',
}
DEFAULT_COLOR = '#888888'

def team_color(name):
    return TEAM_COLORS.get(name, DEFAULT_COLOR)

print('Setup complete')

Setup complete


In [4]:
matches_df    = pd.read_csv('C:\\Users\\pmoh3005\\OneDrive - 7-Eleven, Inc\\Desktop\\development\\7-11apps\\IPL\\archive\\matches.csv')
deliveries_df = pd.read_csv('C:\\Users\\pmoh3005\\OneDrive - 7-Eleven, Inc\\Desktop\\development\\7-11apps\\IPL\\archive\\deliveries.csv')

print(f'matches_df    : {matches_df.shape}')
print(f'deliveries_df : {deliveries_df.shape}')
print(f'\nMatches columns :\n{list(matches_df.columns)}')
matches_df.head()

matches_df    : (1095, 20)
deliveries_df : (260920, 17)

Matches columns :
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [12]:
name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}
print(matches_df["team1"].unique())
print(matches_df["team2"].unique())
print(matches_df["winner"].unique())
# batting_team = np.array([name_map.get(t,t) for t in batting_team])
matches_df["team1"] = matches_df["team1"].replace(name_map)
matches_df["team2"] = matches_df["team2"].replace(name_map)
matches_df["winner"] = matches_df["winner"].replace(name_map)

print("After")
print(matches_df["team1"].unique())
print(matches_df["team2"].unique())
print(matches_df["winner"].unique())

matches_df["season"] = matches_df["season"].replace({"2007/08":"2008","2009/10":"2010","2020/21":"2020"})
matches_df.head()

<StringArray>
['Royal Challengers Bengaluru',                'Punjab Kings',
              'Delhi Capitals',              'Mumbai Indians',
       'Kolkata Knight Riders',            'Rajasthan Royals',
             'Deccan Chargers',         'Chennai Super Kings',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',               'Gujarat Lions',
     'Rising Pune Supergiants',        'Lucknow Super Giants',
              'Gujarat Titans']
Length: 15, dtype: str
<StringArray>
[      'Kolkata Knight Riders',         'Chennai Super Kings',
            'Rajasthan Royals', 'Royal Challengers Bengaluru',
             'Deccan Chargers',                'Punjab Kings',
              'Delhi Capitals',              'Mumbai Indians',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',     'Rising Pune Supergiants',
               'Gujarat Lions',              'Gujarat Titans',
        'Lucknow Super Giants']
Len

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2008,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2008,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2008,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2008,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2008,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [ ]:

matches_df['date']   = pd.to_datetime(matches_df['date'], dayfirst=False, errors='coerce')

In [16]:
matches_df["date"]

0      2008-04-18
1      2008-04-19
2      2008-04-19
3      2008-04-20
4      2008-04-20
          ...    
1090   2024-05-19
1091   2024-05-21
1092   2024-05-22
1093   2024-05-24
1094   2024-05-26
Name: date, Length: 1095, dtype: datetime64[us]

In [18]:
# deliveries wont use but just in case we normalize
for col in ['batting_team','bowling_team']:
    deliveries_df[col] = deliveries_df[col].replace(name_map)
# forgot toss_winnner

matches_df["toss_winner"] = matches_df["toss_winner"].replace(name_map)